# Scanner paper figures — empirical + bootstrap

Empirical, bootstrap-based recreations of two figures from
`scanner_glm_analysis.ipynb`:

1. **Composite-scanner-estimated vs human-confirmed violation rate by benchmark
   and criterion** (one plot per composite scanner).
2. **Scanner flag rate by human-graded severity.**

The GLM notebook produces these from a Bayesian logistic model. Here they are
estimated directly from the data with a stratified-sampling correction and
percentile bootstraps, mirroring the design of `scanner_combined_results.ipynb`.

## Accounting for the validation-sampling design

Human (post-validation) grades exist only for a **subset** of scanned
transcripts, and that subset was **not** drawn uniformly. For every criterion
the validation sample was drawn from the **gpt-5.4** scan and **stratified on
the gpt-5.4 violation flag within each benchmark** — scanner-flagged
transcripts were sampled at a much higher rate than unflagged ones (e.g. for
answer_format / swe_bench, 100% of gpt-flagged vs ~6% of gpt-unflagged
transcripts were validated).

A naive mean over the validated rows therefore over-represents
scanner-flagged transcripts and overstates anything correlated with the flag.
Both figures correct for this with **inverse-probability weighting (IPW)**:
each validated transcript is weighted by `N_pop / n_validated` for its
`(benchmark, gpt-flag)` stratum (the `stratified_ipw_weights` scheme from
`scanner_combined_results.ipynb`), and 95% CIs come from a **stratified
percentile bootstrap** that resamples validated transcripts within each
stratum at fixed size (so the weights are constant across iterations).

- **Figure 1** is drawn **once per composite scanner** (`max` and `floor_mean`,
  each combining the two scanner models' grades). The **dot** is the composite's
  flag rate over the full scanned population (a census, no IPW); the **×** is the
  IPW-corrected `P(human-confirmed violation)` per `(criterion, benchmark)`. The
  × is identical across composites — it is the sampling-adjusted human truth and
  does not depend on the scanner — so the gap between dot and × shows each
  composite's over- vs under-flagging. The IPW weights stay keyed on the gpt-flag
  because that is the sampling design.
- **Figure 2** estimates `P(scanner flags | human ordinal grade)` per
  `(criterion, scanner_model)`, pooled over benchmarks via the per-transcript
  weights. The same gpt-flag×benchmark weights apply to *both* scanner models,
  because they describe the sampling mechanism, not the outcome.

Multiple-choice benchmarks (`gpqa_diamond`, `hle`) are stripped before any
estimate, matching the GLM notebook.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/jeffm/projects/scanner_evaluation")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display
import yaml

from analysis.scan_utils import load_scan_results, load_validations
from analysis.analysis_utils import BENCHMARK_ALIASES, GRADE_LEVELS, shorten_model, make_savers

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 30)

## Config and display invariants

The data selection (criteria, split, scan_ids, validation files, threshold) is
read from the same YAML config used by `scanner_glm_analysis.ipynb`, so the two
notebooks operate on identical data.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "analysis" / "configs" / "glm_test_all.yaml"
with open(CONFIG_PATH) as f:
    _cfg = yaml.safe_load(f)

VIOLATION_THRESHOLD = int(_cfg.get("violation_threshold", 2))
SCANNER_SPECS = {s["target_scanner"]: s for s in (_cfg.get("scanners") or [])}
CRITERIA = sorted(SCANNER_SPECS)

# Multiple-choice benchmarks are stripped before any estimate (match the GLM nb).
EXCLUDE_BENCHMARKS = ["gpqa_diamond", "hle"]

# Every criterion's validation sample was drawn from the gpt-5.4 scan,
# stratified on the gpt-5.4 violation flag; that is the IPW design stratifier.
STRATIFIER_MODEL = "gpt-5.4"
SECONDARY_MODEL = "sonnet-4.6"

# Composite "scanners": each combines the two scanner models' ordinal grades per
# transcript, then binarises at the threshold (same rules as the GLM notebook).
# Figure 1 is drawn once per composite. `max` flags if either model flags (the
# permissive union); `floor_mean` is a conservative average.
COMPOSITES = ["max", "floor_mean"]
COMPOSITE_LABEL = {"max": "max across scanners", "floor_mean": "floor(mean) across scanners"}

RESULTS_DIR = PROJECT_ROOT / "analysis" / "results" / "paper_figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
save_fig, save_table = make_savers(RESULTS_DIR)

# Percentile bootstrap settings.
BOOT_N, BOOT_CI, BOOT_SEED = 2000, 0.95, 0

# Pretty names for axes / legends.
DISPLAY_LABELS = {
    "answer_format": "Answer Format",
    "ground_truth_access": "Ground Truth Access",
    "guessing": "Guessing",
    "tool_failure": "Tool Failure",
    "core_bench": "CORE-Bench",
    "cvebench": "CVE-Bench",
    "gpqa_diamond": "GPQA-Diamond",
    "hle": "HLE",
    "kernelbench": "KernelBench",
    "swe_bench": "SWE-Bench-Verified",
    "tau2_airline": "Tau2-Airline",
    "tau2_retail": "Tau2-Retail",
}

def disp(name):
    return DISPLAY_LABELS.get(name, name)

# Colour per criterion (Figure 1) and per scanner model (Figure 2),
# matching scanner_glm_analysis.ipynb (sonnet = burnt orange, gpt = dark green).
_crit_palette = ["#4477AA", "#AA3377", "#66CCEE", "#CCBB44", "#222255", "#999933"]
CRIT_COLOR = {c: _crit_palette[i % len(_crit_palette)] for i, c in enumerate(CRITERIA)}
MODEL_COLOR = {"sonnet-4.6": "#BF5700", "gpt-5.4": "#1B5E20"}

print(f"CONFIG              = {CONFIG_PATH.relative_to(PROJECT_ROOT)}")
print(f"VIOLATION_THRESHOLD = {VIOLATION_THRESHOLD}")
print(f"CRITERIA            = {CRITERIA}")
print(f"EXCLUDE_BENCHMARKS  = {EXCLUDE_BENCHMARKS}")
print(f"STRATIFIER_MODEL    = {STRATIFIER_MODEL} (validation sampling stratum)")
print(f"COMPOSITES          = {COMPOSITES}")
print(f"RESULTS_DIR         = {RESULTS_DIR.relative_to(PROJECT_ROOT)}")

## Build the transcript-level frame

One row per `(criterion, transcript)`. The **population** for each criterion is
the set of transcripts in the gpt-5.4 scan(s) the validation was actually drawn
from (identified from the `validation_files` names) — that is the universe the
stratified sample represents. Each transcript carries:

- `gpt_grade` / `sonnet_grade` — ordinal scanner grades (max across that
  model's scans), with `gpt_flag` / `sonnet_flag` binarised at the threshold.
- `human_grade` — the post-validation grade where available (`validated` flag),
  with `human_viol` binarised at the threshold.

`gpt_flag` is the sampling stratifier. `sonnet_grade` is looked up by
transcript only so Figure 2 can show the second scanner's flag rate; it never
affects the sampling weights.

In [ ]:
def build_criterion(criterion):
    spec = SCANNER_SPECS[criterion]
    base = PROJECT_ROOT / "evals" / "scans" / criterion / spec["split"]
    raw = load_scan_results(base / "scan-results").copy()
    raw["scan_id"] = raw["scanner_source"].str.removeprefix("scan_id=")
    raw["scanner_model"] = raw["scanner_model"].map(shorten_model)
    raw["benchmark"] = raw["transcript_task_set"].map(lambda b: BENCHMARK_ALIASES.get(b, b))

    keys = sorted(raw["scanner_key"].dropna().unique())
    resolved = spec.get("scanner_key") or (keys[0] if len(keys) == 1 else None)
    if resolved is None:
        raise RuntimeError(f"[{criterion}] multiple scanner_keys {keys}; set scanner_key.")
    raw = raw[raw["scanner_key"] == resolved].copy()

    # gpt-5.4 scan(s) the validation was drawn from = the sampling universe.
    val_stems = [Path(f).stem for f in spec["validation_files"]]
    gpt = raw[raw["scanner_model"] == STRATIFIER_MODEL]
    gpt_val_ids = sorted({sid for sid in gpt["scan_id"].unique()
                          if any(sid in stem for stem in val_stems)})
    pop = (gpt[gpt["scan_id"].isin(gpt_val_ids)]
           .groupby(["transcript_id", "benchmark"], as_index=False)["value_num"]
           .max().rename(columns={"value_num": "gpt_grade"}))

    # Secondary scanner grade for the same transcripts (does not affect weights).
    son = (raw[raw["scanner_model"] == SECONDARY_MODEL]
           .groupby("transcript_id", as_index=False)["value_num"]
           .max().rename(columns={"value_num": "sonnet_grade"}))
    pop = pop.merge(son, on="transcript_id", how="left")

    # Human grade from the selected validation file(s).
    wide = load_validations(base / "validation", prefix="")
    cols = [Path(f).stem for f in spec["validation_files"]]
    long = (wide[["transcript_id"] + cols]
            .melt("transcript_id", value_name="human_grade")[["transcript_id", "human_grade"]])
    long["human_grade"] = pd.to_numeric(long["human_grade"], errors="coerce")
    long = long.dropna(subset=["human_grade"]).drop_duplicates("transcript_id")
    pop = pop.merge(long, on="transcript_id", how="left")

    pop.insert(0, "criterion", criterion)
    print(f"  [{criterion}] universe={len(pop):>4} transcripts from gpt scans "
          f"{gpt_val_ids} · validated={int(pop['human_grade'].notna().sum()):>3}")
    return pop


tx = pd.concat([build_criterion(c) for c in CRITERIA], ignore_index=True)
tx = tx[~tx["benchmark"].isin(EXCLUDE_BENCHMARKS)].copy()

tx["gpt_flag"] = (tx["gpt_grade"] >= VIOLATION_THRESHOLD).astype(int)
tx["sonnet_flag"] = (tx["sonnet_grade"] >= VIOLATION_THRESHOLD).astype(float)
tx["human_viol"] = (tx["human_grade"] >= VIOLATION_THRESHOLD).astype(float)
tx["validated"] = tx["human_grade"].notna()

assert tx["sonnet_grade"].notna().all(), "some population transcripts lack a sonnet grade"

# Composite scanner grades combine the two models' ordinal grades per transcript
# (Figure 1 is drawn once per composite). Both grades are observed for every
# population transcript, so each composite flag is a census just like the
# single-model flag and needs no sampling correction.
_both = tx[["gpt_grade", "sonnet_grade"]]
tx["max_grade"] = _both.max(axis=1)
tx["floor_mean_grade"] = np.floor(_both.mean(axis=1))
for _comp in COMPOSITES:
    tx[f"{_comp}_flag"] = (tx[f"{_comp}_grade"] >= VIOLATION_THRESHOLD).astype(int)

print()
print(f"Transcript frame: {len(tx):,} rows · {tx['criterion'].nunique()} criteria")
display(tx.groupby("criterion").agg(
    n_universe=("transcript_id", "size"),
    n_validated=("validated", "sum"),
    benchmarks=("benchmark", "nunique"),
))

## IPW weights and the bootstrap helper

For each `(criterion, benchmark, gpt_flag)` stratum, the inverse-probability
weight is `N_population / n_validated`. A validated transcript stands in for
that many population transcripts in its stratum. A stratum that exists in the
population but has **no** validated transcript cannot be reweighted; Figure 1
drops any benchmark with such a gap rather than silently extrapolating.

In [ ]:
pop_counts = tx.groupby(["criterion", "benchmark", "gpt_flag"]).size().rename("N_pop")
val_counts = (tx[tx["validated"]].groupby(["criterion", "benchmark", "gpt_flag"]).size()
              .rename("n_val"))
strata = pd.concat([pop_counts, val_counts], axis=1).reset_index()
strata["n_val"] = strata["n_val"].fillna(0).astype(int)
strata["weight"] = np.where(strata["n_val"] > 0, strata["N_pop"] / strata["n_val"], np.nan)
strata["samp_frac"] = strata["n_val"] / strata["N_pop"]

tx = tx.merge(strata[["criterion", "benchmark", "gpt_flag", "weight", "N_pop", "n_val"]],
              on=["criterion", "benchmark", "gpt_flag"], how="left")

uncovered = strata[strata["n_val"] == 0]
if not uncovered.empty:
    print("Population strata with no validated transcript (excluded from Fig 1):")
    display(uncovered[["criterion", "benchmark", "gpt_flag", "N_pop"]])

display(strata.round(3))


def wrate(indicator, weights):
    """IPW-weighted mean of a 0/1 indicator."""
    indicator = np.asarray(indicator, dtype=float)
    weights = np.asarray(weights, dtype=float)
    tot = weights.sum()
    return float((indicator * weights).sum() / tot) if tot > 0 else float("nan")


def stratified_bootstrap(val_df, strata_cols, stat_fn,
                         n_boot=BOOT_N, ci=BOOT_CI, seed=BOOT_SEED):
    """Stratified percentile bootstrap.

    Resamples validated rows with replacement *within* each stratum
    (``strata_cols``) at fixed size, so the IPW weights are constant across
    iterations. ``stat_fn(frame) -> dict[str, float]`` is evaluated on the
    full data (point estimate) and each resample; returns
    ``{name: (point, lo, hi)}``.
    """
    d = val_df.reset_index(drop=True)
    groups = [g.index.to_numpy() for _, g in d.groupby(strata_cols, dropna=False)]
    point = stat_fn(d)
    samp = {k: np.full(n_boot, np.nan) for k in point}
    rng = np.random.default_rng(seed)
    for b in range(n_boot):
        sel = np.concatenate([rng.choice(idx, size=idx.size, replace=True) for idx in groups])
        boot = stat_fn(d.iloc[sel])
        for k, v in boot.items():
            samp[k][b] = v
    alpha = (1.0 - ci) / 2.0
    out = {}
    for k, p in point.items():
        arr = samp[k][~np.isnan(samp[k])]
        if arr.size and not np.isnan(p):
            lo, hi = float(np.quantile(arr, alpha)), float(np.quantile(arr, 1 - alpha))
            # Clip the percentile interval to include the point estimate (it can
            # fall outside for boundary rates / small weighted strata), matching
            # the convention used in scanner_glm_analysis.ipynb.
            out[k] = (p, min(p, lo), max(p, hi))
        else:
            out[k] = (p, float("nan"), float("nan"))
    return out

## Figure 1 — Composite-scanner estimated vs human-confirmed violation rate

Drawn **once per composite scanner** — a single "scanner" formed by combining
the two scanner models' ordinal grades per transcript, then binarising at the
threshold:

- **`max` (either flags)** — `max(gpt_grade, sonnet_grade)`; flags if *either*
  model flags. The permissive union.
- **`floor_mean`** — `⌊mean(gpt_grade, sonnet_grade)⌋`; a conservative average.

Per `(criterion, benchmark)`:

- **Dot + 95% CI** = the **composite scanner-estimated violation rate** — the
  fraction of transcripts the composite flags. Both models' grades are observed
  for *every* transcript in the scanned population, so the composite flag is a
  census and needs **no** sampling correction: it is computed directly over the
  full population, with a plain percentile-bootstrap CI (resampling the
  population transcripts → benchmark-level sampling uncertainty).
- **× mark** = the **human-confirmed violation rate**, IPW-corrected for the
  stratified validation sampling (the quantity the GLM figure estimates). This
  reference is **identical across the two composite plots** — it is the
  sampling-adjusted human truth and does not depend on which scanner is used.
  Human grades exist only on the gpt-flag-stratified subset, so this one *does*
  carry the gpt-flag IPW weights.

Clustered by benchmark, criteria side by side in colour. The gap between the dot
and its × shows how the composite's flag rate compares to the sampling-adjusted
human truth (over- vs under-flagging). The IPW weights stay keyed on the
**gpt-5.4** flag (and the human-coverage check uses gpt-flag strata) because the
validation sampling was built on it, regardless of which composite is plotted.

In [ ]:
def figure1_for_composite(comp):
    """Figure 1 for one composite scanner (``comp`` -> ``f"{comp}_flag"``).

    The dot is the composite flag rate over the full population (a census, plain
    bootstrap). The × is the human-confirmed rate, IPW-corrected on the gpt-flag
    sampling strata — identical for every composite. Returns the rate table.
    """
    flag_col = f"{comp}_flag"
    label = COMPOSITE_LABEL[comp]

    viol_rows = []
    for (crit, bench), g in tx.groupby(["criterion", "benchmark"]):
        # Composite scanner-estimated violation rate: the composite flag is
        # observed for the whole population, so compute directly over it (no IPW)
        # with a plain bootstrap CI.
        scan = stratified_bootstrap(
            g, ["benchmark"],  # single group -> ordinary bootstrap over all pop rows
            lambda d: {"rate": float(d[flag_col].mean())},
        )
        s_rate, s_lo, s_hi = scan["rate"]

        # Human-confirmed violation rate: observed only on the stratified subset,
        # so IPW-corrected (gpt-flag strata — the sampling design). Drawn as the ×.
        v = g[g["validated"]]
        present = set(g["gpt_flag"].unique())
        covered = bool(present) and all((v["gpt_flag"] == s).sum() > 0 for s in present)
        if not v.empty and covered:
            hh = stratified_bootstrap(
                v, ["gpt_flag"],
                lambda d: {"rate": wrate(d["human_viol"], d["weight"])},
            )
            h_rate, h_lo, h_hi = hh["rate"]
            naive = float(v["human_viol"].mean())
        else:
            h_rate = h_lo = h_hi = naive = float("nan")
            if not v.empty:
                print(f"  {crit}/{bench}: no IPW human rate (a gpt stratum lacks validation)")

        viol_rows.append({
            "criterion": crit, "benchmark": bench,
            "n_pop": int(len(g)), "n_val": int(len(v)),
            "scanner_rate": s_rate, "scanner_lo": s_lo, "scanner_hi": s_hi,
            "human_rate_ipw": h_rate, "human_lo": h_lo, "human_hi": h_hi,
            "human_rate_naive": naive,
        })

    violation_rates = (pd.DataFrame(viol_rows)
                       .sort_values(["benchmark", "criterion"]).reset_index(drop=True))
    save_table(violation_rates, f"violation_rates_empirical_{comp}")
    print(f"Composite scanner: {label}")
    display(violation_rates.round(3))

    benchmarks_all = sorted(violation_rates["benchmark"].unique())
    crit_offsets = np.linspace(-0.27, 0.27, len(CRITERIA))

    fig, ax = plt.subplots(figsize=(7.8, 4.6))
    for off, crit in zip(crit_offsets, CRITERIA):
        sm = (violation_rates[violation_rates["criterion"] == crit]
              .set_index("benchmark").reindex(benchmarks_all))
        xpos = np.arange(len(benchmarks_all)) + off
        color = CRIT_COLOR[crit]
        ax.errorbar(
            xpos, sm["scanner_rate"],
            yerr=[np.clip(sm["scanner_rate"] - sm["scanner_lo"], 0, None),
                  np.clip(sm["scanner_hi"] - sm["scanner_rate"], 0, None)],
            fmt="o", color=color, ms=6, lw=1.8, capsize=3, label=disp(crit),
        )
        ax.scatter(xpos, sm["human_rate_ipw"], marker="x", color=color, s=36, zorder=3, alpha=0.85)

    ax.set_xticks(np.arange(len(benchmarks_all)))
    ax.set_xticklabels([disp(b) for b in benchmarks_all], rotation=20, ha="right")
    for i in range(len(benchmarks_all)):
        if i % 2:
            ax.axvspan(i - 0.5, i + 0.5, color="#f2f2f2", zorder=0)
    ax.set_xlim(-0.5, len(benchmarks_all) - 0.5)
    ax.set_ylim(0, 1)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#dddddd", lw=0.6)
    ax.set_axisbelow(True)
    ax.set_ylabel("Violation rate")
    ax.set_title(f"Composite scanner ({label}) — estimated and "
                 "human-confirmed violation rate by benchmark and criterion")
    legend_handles = [
        Line2D([], [], color=CRIT_COLOR[c], marker="o", lw=1.8, label=disp(c))
        for c in CRITERIA
    ] + [
        Line2D([], [], color="#555555", marker="o", ls="", label=f"composite flag rate ({label})"),
        Line2D([], [], color="#555555", marker="x", ls="", label="human-confirmed (IPW-adj.)"),
    ]
    ax.legend(handles=legend_handles, frameon=False, fontsize=8, loc="upper right")
    fig.tight_layout()
    save_fig(fig, f"violation_rates_by_benchmark_{comp}")
    plt.show()
    return violation_rates


# One separate plot per composite scanner (max, then floor_mean).
violation_rates_by_composite = {comp: figure1_for_composite(comp) for comp in COMPOSITES}

## Figure 2 — Scanner flag rate by human-graded severity

`P(scanner flags | human ordinal grade)` per `(criterion, scanner_model)`,
pooled across benchmarks using the per-transcript IPW weights, with a
stratified-bootstrap 95% CI. One panel per criterion; the two scanner models
are coloured (sonnet = burnt orange, gpt = dark green). The dashed line is the
binarisation threshold (grades to its right are human-confirmed violations);
marker area scales with the number of validated transcripts at that grade.

Because the validation sample was stratified on the **gpt** flag, the
correction is largest for the gpt series (its positives were directly
over-sampled) and smaller — but still present, via scanner agreement — for the
sonnet series.

In [ ]:
display_models = sorted(MODEL_COLOR)  # ['gpt-5.4', 'sonnet-4.6']
human_grades = sorted(int(g) for g in tx.loc[tx["validated"], "human_grade"].unique())
model_cols = {"gpt-5.4": "gpt_flag", "sonnet-4.6": "sonnet_flag"}

sev_rows = []
for crit in CRITERIA:
    v = tx[(tx["criterion"] == crit) & tx["validated"]].copy()
    v["hg_int"] = v["human_grade"].astype(int)
    for model in display_models:
        col = model_cols[model]

        def stat(d, col=col):
            out = {}
            for hg in human_grades:
                sub = d[d["hg_int"] == hg]
                out[f"g{hg}"] = wrate(sub[col], sub["weight"]) if len(sub) else np.nan
            return out

        res = stratified_bootstrap(v, ["benchmark", "gpt_flag"], stat)
        for hg in human_grades:
            n = int((v["hg_int"] == hg).sum())
            if n == 0:
                continue
            rate, lo, hi = res[f"g{hg}"]
            sev_rows.append({
                "criterion": crit, "scanner_model": model, "human_grade": hg,
                "n": n, "flag_rate": rate, "lo": lo, "hi": hi,
            })

severity = pd.DataFrame(sev_rows)
save_table(severity, "flag_rate_by_severity_empirical")
display(severity.round(3))

n_panels = len(CRITERIA)
n_cols = 2
n_rows = (n_panels + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(11.5, 3.8 * n_rows),
                         sharey=True, squeeze=False)
for ax, crit in zip(axes.ravel(), CRITERIA):
    for off, m in zip((-0.08, 0.08), display_models):
        sm = (severity[(severity["criterion"] == crit) & (severity["scanner_model"] == m)]
              .sort_values("human_grade"))
        if sm.empty:
            continue
        color = MODEL_COLOR[m]
        x = sm["human_grade"] + off
        ax.errorbar(x, sm["flag_rate"],
                    yerr=[sm["flag_rate"] - sm["lo"], sm["hi"] - sm["flag_rate"]],
                    fmt="none", ecolor=color, elinewidth=1.1, alpha=0.6)
        ax.plot(x, sm["flag_rate"], "-", color=color, lw=1.2, alpha=0.7)
        ax.scatter(x, sm["flag_rate"], s=18 + 2.2 * sm["n"], color=color, zorder=3, label=m)
    ax.axvline(VIOLATION_THRESHOLD - 0.5, color="#999999", lw=1, ls="--")
    ax.set_title(disp(crit))
    ax.set_xticks(human_grades)
    ax.set_ylim(-0.04, 1.04)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e8e8e8", lw=0.6)
    ax.set_axisbelow(True)

for ax in axes.ravel()[n_panels:]:
    ax.axis("off")
axes[0][0].legend(frameon=False, fontsize=9, loc="upper left")
for ax in axes[-1]:
    ax.set_xlabel("Human Grade")
for ax in axes[:, 0]:
    ax.set_ylabel("P(scanner flag), 95% CI")
fig.suptitle("Scanner flag rate by human-graded severity")
fig.tight_layout(rect=[0, 0, 1, 0.97])
save_fig(fig, "flag_rate_by_severity")
plt.show()